# Latih detektor tingkat kematangan strawberry (YOLO26) di Kaggle

Notebook ini terpisah dari notebook penyakit (`strawberry_kaggle (2).ipynb`) dan menghasilkan model **kedua**, khusus kematangan buah, dengan dua kelas: `ripe` (matang) dan `unripe` (mentah). Model penyakit tidak diubah. Realtime dijalankan di laptop dengan `kematangan/realtime_kematangan.py` memakai `best.pt` dari notebook ini.

Di luar dua kelas dari model, `realtime_kematangan.py` juga menghitung **persentase warna merah** pada tiap buah, lalu menjadikannya tingkat: *mentah*, *setengah matang*, atau *matang*. Dataset ini tidak punya label "setengah matang", jadi tingkat tengah itu hanya berasal dari warna, bukan dari model. Sel 8 memeriksa seberapa cocok perkiraan warna itu dengan label asli.

**Atur dulu di panel Settings (sisi kanan), sebelum menjalankan sel apa pun:**

1. **Accelerator**: GPU T4 x2 (skrip hanya memakai satu GPU). Kalau muncul error CUDA "no kernel image", ganti ke T4.
2. **Internet**: On, untuk `pip install` dan mengunduh bobot awal `yolo26s.pt`.
3. **Add Input**:
   - **wajib**: dataset kematangan `strawberry.v1i.yolov8.zip` (Roboflow aliazz/strawberry-fowax, kelas `ripe`/`unripe`). Create → New Dataset, unggah zip-nya apa adanya.
   - **opsional, disarankan**: paket `strawberry_extra` yang sudah Anda unggah untuk notebook penyakit. Dari paket itu hanya foto `neg_leaf_*` (daun strawberry) dan `neg_otherplant_*` (tanaman lain) yang dipakai, sebagai gambar tanpa buah, supaya model tidak menandai daun atau tanaman lain sebagai buah.
   - **jangan** lampirkan `strawberry_extra_v2` dan `benchmark_healthy`: isinya foto dari dataset kematangan ini juga, yang labelnya dihapus. Kalaupun terlampir, notebook mengabaikannya.

Sel 3 membagi ulang train/val/test. Dataset ini berisi banyak salinan dari foto yang sama (sekitar 950 foto asli untuk 1.650 gambar train), dan pembagian bawaannya hanya 99 gambar val dan 21 gambar test. Semua salinan satu foto dimasukkan ke split yang sama, supaya skor test tidak terlalu optimis.

Lalu jalankan sel dari atas ke bawah. Untuk training panjang, pakai **Save Version → Save & Run All** supaya tetap berjalan walau tab ditutup.

In [ ]:
# 1) Pasang Ultralytics dan cek GPU
# Jika muncul error saat import setelah pip install, restart kernel lalu lanjut dari sel ini.
!pip install -q -U ultralytics

import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU TIDAK aktif -> Settings > Accelerator > pilih GPU, lalu jalankan ulang dari sel ini.")

In [ ]:
# 2) Cari dataset kematangan (+ gambar negatif dari paket strawberry_extra, jika dilampirkan)
import collections, os, re
from pathlib import Path
import yaml

DATA_BASES = ("/kaggle/input", ".")
# Urutan = ID kelas model ini. Nama kelas dataset sumber dipetakan ke sini (huruf besar/kecil diabaikan).
# Dataset lain boleh ikut dilampirkan asal semua kelasnya ada di NAME_MAP.
CLASS_NAMES = ["ripe", "unripe"]
NAME_MAP = {"ripe": "ripe", "matang": "ripe", "unripe": "unripe", "mentah": "unripe"}
USE_NEGATIVES = True
NEG_PREFIXES = ("neg_leaf_", "neg_otherplant_")  # JANGAN tambah neg_ripeness*: itu foto dataset ini juga, labelnya dihapus
SPLIT_DIRS = {"train": "train", "valid": "val", "val": "val", "test": "test"}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def read_names(root):
    names = (yaml.safe_load((root / "data.yaml").read_text(encoding="utf-8")) or {}).get("names") or []
    return [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)

def find_roots(bases):
    found = set()
    for base in bases:
        for dirpath, _, _ in os.walk(base, followlinks=True):
            root = Path(dirpath)
            if (root / "data.yaml").is_file() and any((root / s / "images").is_dir() for s in SPLIT_DIRS):
                found.add(root.resolve())
    return sorted(found)

def yolo_box(parts):
    """Satu baris label YOLO -> (kelas, xc, yc, w, h). Baris poligon (segmentasi) diubah menjadi kotak."""
    c, v = int(float(parts[0])), [float(x) for x in parts[1:]]
    if len(v) == 4:
        return (c, *v)
    xs, ys = v[0::2], v[1::2]
    return c, (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)

ITEMS = []  # satu entri per gambar: path sumber, kotak (kelas sudah dipetakan), jenis, split bawaan
seen = set()
for root in find_roots(DATA_BASES):
    names = read_names(root)
    mapped = [NAME_MAP.get(str(n).strip().lower()) for n in names]
    kind = "kematangan" if names and all(mapped) else ("negatif" if USE_NEGATIVES else None)
    remap = {i: CLASS_NAMES.index(m) for i, m in enumerate(mapped) if m}
    n_img, bad = 0, 0
    for sd, split in SPLIT_DIRS.items():
        idir = root / sd / "images"
        if kind is None or not idir.is_dir():
            continue
        for p in sorted(idir.iterdir()):
            if p.suffix.lower() not in IMG_EXT or (p.name, p.stat().st_size) in seen:
                continue  # dataset yang sama terlampir dua kali
            if kind == "negatif":
                if not p.name.startswith(NEG_PREFIXES):
                    continue
                boxes = []
            else:
                lp = root / sd / "labels" / (p.stem + ".txt")
                rows = [yolo_box(l.split()) for l in (lp.read_text(encoding="utf-8").splitlines() if lp.is_file() else []) if l.strip()]
                bad += sum(c not in remap for c, *_ in rows)
                boxes = [(remap[c], *b) for c, *b in rows if c in remap]
            seen.add((p.name, p.stat().st_size))
            ITEMS.append({"src": p, "boxes": boxes, "kind": kind, "orig": split})
            n_img += 1
    if n_img:
        print(f"{kind:10s} {root}: {n_img} gambar" + (f" | kelas {names} -> {mapped}" if kind == "kematangan" else ""))
        if bad:
            print(f"  PERINGATAN: {bad} kotak dengan ID kelas di luar data.yaml dibuang")
    else:
        print(f"DILEWATI   {root}: kelas {names}")

if not any(it["kind"] == "kematangan" for it in ITEMS):
    raise FileNotFoundError(
        "Dataset kematangan tidak ditemukan. Lampirkan strawberry.v1i.yolov8.zip lewat Add Input, lalu jalankan ulang sel ini. "
        f"Nama kelas yang dikenali: {sorted(NAME_MAP)}"
    )
n_box = collections.Counter(b[0] for it in ITEMS for b in it["boxes"])
n_imgcls = collections.Counter(c for it in ITEMS for c in {b[0] for b in it["boxes"]})
print(f"\nTotal {len(ITEMS)} gambar ({sum(it['kind'] == 'negatif' for it in ITEMS)} negatif, "
      f"{sum(it['kind'] == 'kematangan' and not it['boxes'] for it in ITEMS)} foto kematangan tanpa kotak)")
for k, name in enumerate(CLASS_NAMES):
    print(f"  {name:8s}: {n_box[k]} kotak di {n_imgcls[k]} gambar")

In [ ]:
# 3) Bagi ulang train/val/test tanpa gambar kembar
# Roboflow memberi nama <nama asli>_jpg.rf.<hash>.jpg, jadi salinan foto yang sama dikenali dari nama aslinya.
# Foto yang mirip (termasuk versi cermin/putar) juga dikelompokkan lewat hash gambar. Lalu pembagian dilakukan
# per KELOMPOK, dengan proporsi tiap kelas dijaga.
import csv, time
import numpy as np
from PIL import Image

SPLIT_RATIOS = {"train": 0.65, "val": 0.15, "test": 0.20}
SPLIT_SEED = 0
DUP_T = 20  # jarak hash (dari 256 bit) <= ini: dianggap foto yang sama
SPLIT_NAMES = ("train", "val", "test")
NC = len(CLASS_NAMES)

_TR = getattr(Image, "Transpose", Image)
_TRANS = [None, _TR.FLIP_LEFT_RIGHT, _TR.FLIP_TOP_BOTTOM, _TR.ROTATE_90, _TR.ROTATE_180, _TR.ROTATE_270, _TR.TRANSPOSE, _TR.TRANSVERSE]

def source_key(p):
    return re.sub(r"\.rf\.[0-9a-f]+$", "", p.stem)

def hash8(path):
    """dHash 256-bit dari 8 orientasi (asli, cermin, rotasi) supaya versi yang dibalik/diputar ikut terdeteksi."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((640, 640))
        out = []
        for t in _TRANS:
            a = np.asarray((im if t is None else im.transpose(t)).convert("L").resize((17, 16), Image.BILINEAR), dtype=np.int16)
            out.append((a[:, 1:] > a[:, :-1]).ravel())
    return np.stack(out)

def group_duplicates(paths):
    n = len(paths)
    t0 = time.time()
    H = np.stack([hash8(p) for p in paths]).astype(np.float32) * 2 - 1  # (n, 8, 256), nilai +-1
    D = np.full((n, n), 256, dtype=np.float32)
    for t in range(H.shape[1]):
        np.minimum(D, (256 - H[:, t] @ H[:, 0].T) / 2, out=D)  # jarak Hamming terkecil antar orientasi
    D = np.minimum(D, D.T)
    print(f"Hash {n} gambar: {time.time() - t0:.0f} detik")
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in zip(*np.nonzero(np.triu(D <= DUP_T, 1))):
        parent[find(a)] = find(b)
    first = {}
    for i, p in enumerate(paths):  # salinan Roboflow dari foto asli yang sama
        parent[find(i)] = find(first.setdefault(source_key(p), i))
    groups = collections.defaultdict(list)
    for i in range(n):
        groups[find(i)].append(i)
    return list(groups.values())

def stratified_group_split(groups, cls_sets, nc, ratios, seed):
    """Membagi kelompok ke train/val/test; kelompok berisi kelas paling langka dibagi lebih dulu."""
    K = nc + 1  # indeks terakhir = gambar tanpa kotak
    vec = []
    for g in groups:
        v = np.zeros(K)
        for i in g:
            for k in (cls_sets[i] or {nc}):
                v[k] += 1
        vec.append(v)
    total = np.sum(vec, axis=0)
    target = {s: ratios[s] * total for s in SPLIT_NAMES}
    count = {s: np.zeros(K) for s in SPLIT_NAMES}
    n_img = {s: 0 for s in SPLIT_NAMES}
    n_all = sum(len(g) for g in groups)
    rng = np.random.default_rng(seed)
    tie = rng.random(len(groups))
    order = sorted(range(len(groups)), key=lambda j: (min(total[k] for k in np.nonzero(vec[j])[0]), -len(groups[j]), tie[j]))
    assign = {}
    for j in order:
        v = vec[j]
        def score(s):
            need = sum(v[k] * (target[s][k] - count[s][k]) / max(target[s][k], 1e-9) for k in np.nonzero(v)[0])
            return (need, ratios[s] * n_all - n_img[s])
        s = max(SPLIT_NAMES, key=score)
        count[s] += v
        n_img[s] += len(groups[j])
        for i in groups[j]:
            assign[i] = s
    return assign

paths = [it["src"] for it in ITEMS]
cls_sets = [frozenset(b[0] for b in it["boxes"]) for it in ITEMS]
groups = group_duplicates(paths)
multi = [g for g in groups if len(g) > 1]
print(f"{len(ITEMS)} gambar -> {len(groups)} kelompok | berisi >1 gambar: {len(multi)} ({sum(map(len, multi))} gambar, terbesar {max(map(len, groups))})")
for s in ("val", "test"):
    own = [i for i, it in enumerate(ITEMS) if it["orig"] == s]
    leak = sum(1 for g in groups for i in g if ITEMS[i]["orig"] == s and any(ITEMS[x]["orig"] == "train" for x in g))
    if own:
        print(f"Pembagian bawaan: {leak} dari {len(own)} gambar {s} ({leak / len(own):.0%}) punya kembaran di train")

assign = stratified_group_split(groups, cls_sets, NC, SPLIT_RATIOS, SPLIT_SEED)
assert all(len({assign[i] for i in g}) == 1 for g in groups), "masih ada kelompok yang terbagi ke beberapa split"

def per_class(idx):
    c = collections.Counter()
    for i in idx:
        for k in (cls_sets[i] or {NC}):
            c[k] += 1
    return c
old = {s: per_class([i for i, it in enumerate(ITEMS) if it["orig"] == s]) for s in SPLIT_NAMES}
new = {s: per_class([i for i in range(len(ITEMS)) if assign[i] == s]) for s in SPLIT_NAMES}
print(f"\n{'jumlah gambar per kelas':24s} {'train':>11s} {'val':>9s} {'test':>9s}   (lama -> baru)")
for k, name in enumerate(CLASS_NAMES + ["(tanpa buah)"]):
    print(f"{name:24s} " + " ".join(f"{old[s][k]:4d}->{new[s][k]:<4d}" for s in SPLIT_NAMES))
print(f"{'TOTAL':24s} " + " ".join(f"{sum(it['orig'] == s for it in ITEMS):4d}->{sum(a == s for a in assign.values()):<4d}" for s in SPLIT_NAMES))

with open("split_assignment.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["image", "jenis", "split_lama", "split_baru", "kelompok"])
    gid = {i: n for n, g in enumerate(groups) for i in g}
    for i, it in enumerate(ITEMS):
        w.writerow([str(it["src"]), it["kind"], it["orig"], assign[i], gid[i]])
print("\nPembagian disimpan di split_assignment.csv. Hasilnya selalu sama selama data dan SPLIT_SEED sama.")

In [ ]:
# 4) Salin ke /kaggle/working/data dengan nama pendek, lalu buat data.yaml
# Dataset di /kaggle/input hanya-baca, sedangkan ID kelas perlu dipetakan ulang, jadi gambar dan label disalin.
import math, random, shutil

WORK = Path("data").resolve()
shutil.rmtree(WORK, ignore_errors=True)
for s in SPLIT_NAMES:
    (WORK / s / "images").mkdir(parents=True)
    (WORK / s / "labels").mkdir(parents=True)
SPLITS = {s: [] for s in SPLIT_NAMES}
CLS_OF = {}
with open("data_manifest.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["file", "split", "sumber"])
    for i, it in enumerate(ITEMS):
        s = assign[i]
        prefix = "rp" if it["kind"] == "kematangan" else it["src"].name.rsplit("_", 1)[0]  # neg_leaf / neg_otherplant
        dst = WORK / s / "images" / f"{prefix}_{i:05d}{it['src'].suffix.lower()}"
        shutil.copyfile(it["src"], dst)
        (WORK / s / "labels" / f"{dst.stem}.txt").write_text(
            "".join(f"{c} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n" for c, x, y, bw, bh in it["boxes"]), encoding="utf-8")
        SPLITS[s].append(dst)
        CLS_OF[dst] = cls_sets[i]
        w.writerow([dst.name, s, str(it["src"])])

USE_RFS = True  # ulang gambar berisi buah mentah (kelas yang lebih sedikit)
RFS_T = 0.5     # makin besar = makin banyak pengulangan kelas langka

def build_rfs_list(train_paths, nc, t, seed=0):
    """Repeat factor sampling: gambar yang memuat kelas langka diulang agar kelas itu lebih sering dilihat model."""
    labeled = [CLS_OF[p] for p in train_paths if CLS_OF[p]]
    freq = [sum(k in c for c in labeled) / len(labeled) for k in range(nc)]
    rep = [max(1.0, math.sqrt(t / f)) if f > 0 else 1.0 for f in freq]
    rng = random.Random(seed)
    lines = []
    for p in train_paths:
        rf = max((rep[k] for k in CLS_OF[p]), default=1.0)
        lines += [str(p)] * (int(rf) + (rng.random() < rf - int(rf)))  # pembulatan stokastik
    return lines, freq, rep

if USE_RFS:
    lines, freq, rep = build_rfs_list(SPLITS["train"], NC, RFS_T)
    print(f"RFS: {len(SPLITS['train'])} gambar train -> {len(lines)} entri per epoch")
    for k, name in enumerate(CLASS_NAMES):
        print(f"  {name:8s} ada di {freq[k] * 100:5.1f}% gambar berlabel | diulang {rep[k]:.2f}x")
    train_ref = Path("train_list.txt").resolve()
    train_ref.write_text("\n".join(lines) + "\n", encoding="utf-8")
else:
    lines, train_ref = SPLITS["train"], WORK / "train" / "images"
N_TRAIN = len(lines)

cfg = {"train": str(train_ref), "val": str(WORK / "val" / "images"), "test": str(WORK / "test" / "images"),
       "names": dict(enumerate(CLASS_NAMES))}
DATA_YAML = Path("data.yaml").resolve()
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print({s: len(SPLITS[s]) for s in SPLIT_NAMES}, "gambar | train per epoch:", N_TRAIN)
print(DATA_YAML.read_text())

In [ ]:
# 5) Training
import os
os.environ["WANDB_MODE"] = "disabled"  # cegah Kaggle meminta API key Weights & Biases

from ultralytics import YOLO

MODEL   = "yolo26s.pt"  # yolo26n.pt = paling ringan, yolo26m.pt = lebih akurat
EPOCHS  = 100
IMGSZ   = 640           # gambar dataset 640x640
BATCH   = 16
WORKERS = 4             # turunkan ke 2 jika muncul error DataLoader / shared memory
RUN     = "ripeness"
HSV_H   = 0.005         # geser warna (hue) sedikit saja: warna adalah tanda utama kematangan (default Ultralytics 0.015)

model = YOLO(MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    cache=True,    # simpan gambar di RAM agar epoch lebih cepat (dataset kecil)
    patience=30,   # berhenti lebih awal kalau val tidak membaik 30 epoch
    flipud=0.5,    # buah tidak punya orientasi atas-bawah yang tetap
    hsv_h=HSV_H,
    name=RUN,
)
BEST = Path(model.trainer.best)
print("Bobot terbaik:", BEST)

In [ ]:
# 6) Evaluasi akhir pada split test (jangan dipakai untuk tuning; lihat hanya setelah training selesai)
best = YOLO(str(BEST))
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH)
print(f"\n[TEST] mAP50-95 = {metrics.box.map:.3f} | mAP50 = {metrics.box.map50:.3f} | P = {metrics.box.mp:.3f} | R = {metrics.box.mr:.3f}")
print(f"{'kelas':8s} {'P':>6s} {'R':>6s} {'AP50':>6s} {'AP50-95':>8s}")
PER_CLASS = {}
for k, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(k)
    PER_CLASS[CLASS_NAMES[int(c)]] = {"P": round(float(p), 4), "R": round(float(r), 4), "AP50": round(float(ap50), 4), "AP50_95": round(float(ap), 4)}
    print(f"{CLASS_NAMES[int(c)]:8s} {p:6.3f} {r:6.3f} {ap50:6.3f} {ap:8.3f}")

# Seberapa sering matang dan mentah tertukar (jumlah kotak, conf >= 0.25)
CONFUSION = None
cm = getattr(getattr(metrics, "confusion_matrix", None), "matrix", None)
if cm is not None:
    lab = CLASS_NAMES + ["latar"]
    CONFUSION = {f"pred_{lab[r]}": {f"asli_{lab[c]}": int(cm[r][c]) for c in range(len(lab))} for r in range(len(lab))}
    print("\nConfusion matrix (baris = prediksi, kolom = sebenarnya; 'latar' = tidak ada buah / buah terlewat)")
    print(" " * 14 + "".join(f"{l:>9s}" for l in lab))
    for r, l in enumerate(lab):
        print(f"pred {l:9s}" + "".join(f"{int(cm[r][c]):9d}" for c in range(len(lab))))

In [ ]:
# 7) Alarm palsu: berapa gambar tanpa buah (neg_leaf_*, neg_otherplant_*) di split test yang tetap memicu deteksi?
import matplotlib.pyplot as plt

FP_RATE = None
neg = sorted(p for p in SPLITS["test"] if p.name.startswith("neg_"))
if not neg:
    print("Tidak ada gambar negatif di split test. Lampirkan paket strawberry_extra untuk mengukur alarm palsu.")
else:
    CONF = 0.4  # sama dengan default realtime_kematangan.py
    preds = best.predict([str(p) for p in neg], imgsz=IMGSZ, conf=CONF, verbose=False)
    hits = [(p, r) for p, r in zip(neg, preds) if len(r.boxes)]
    FP_RATE = len(hits) / len(neg)
    print(f"Gambar tanpa buah di test: {len(neg)} | memicu deteksi (conf >= {CONF}): {len(hits)} ({FP_RATE:.1%})")
    for src in sorted({p.name.rsplit("_", 1)[0] for p in neg}):
        own = [(p, r) for p, r in zip(neg, preds) if p.name.startswith(src + "_")]
        print(f"  {src:16s}: {sum(len(r.boxes) > 0 for _, r in own)} dari {len(own)}")
    if hits:
        fig, axes = plt.subplots(2, 4, figsize=(18, 9))
        for ax in axes.ravel():
            ax.axis("off")
        for ax, (p, r) in zip(axes.ravel(), hits[:8]):
            ax.imshow(r.plot()[:, :, ::-1])
            ax.set_title(p.name, fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
# 8) Cek perkiraan warna: persentase piksel merah di tengah kotak, dihitung pada kotak label ASLI di val + test.
#    realtime_kematangan.py memakai rumus yang sama untuk tingkat mentah / setengah matang / matang.
#    Kalau median 'ripe' dan 'unripe' berjauhan, ambang di bawah masuk akal; kalau tidak, andalkan kelas dari model saja.
import cv2

RED_LEVELS = (0.3, 0.7)  # < 0.3 mentah, 0.3-0.7 setengah matang, >= 0.7 matang (sama dengan --levels di realtime_kematangan.py)

def red_ratio(crop):
    """Fraksi piksel merah di elips tengah kotak (BGR). None jika kotak terlalu kecil."""
    h, w = crop.shape[:2]
    if h < 6 or w < 6:
        return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    mask = np.zeros((h, w), np.uint8)
    cv2.ellipse(mask, (w // 2, h // 2), (max(1, int(w * 0.35)), max(1, int(h * 0.35))), 0, 0, 360, 255, -1)
    H, S, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    fruit = (mask > 0) & (V >= 40)  # abaikan bayangan yang terlalu gelap
    n = int(fruit.sum())
    if n == 0:
        return None
    red = fruit & ((H <= 10) | (H >= 160)) & (S >= 90)
    return float(red.sum()) / n

def level_of(r):
    return "mentah" if r < RED_LEVELS[0] else ("setengah matang" if r < RED_LEVELS[1] else "matang")

ratios = collections.defaultdict(list)
for p in SPLITS["val"] + SPLITS["test"]:
    lp = p.parent.parent / "labels" / f"{p.stem}.txt"
    rows = [l.split() for l in lp.read_text(encoding="utf-8").splitlines() if l.strip()]
    if not rows:
        continue
    img = cv2.imread(str(p))
    ih, iw = img.shape[:2]
    for c, x, y, bw, bh in ((int(r[0]), *map(float, r[1:5])) for r in rows):
        x0, x1 = int((x - bw / 2) * iw), int((x + bw / 2) * iw)
        y0, y1 = int((y - bh / 2) * ih), int((y + bh / 2) * ih)
        r = red_ratio(img[max(0, y0):y1, max(0, x0):x1])
        if r is not None:
            ratios[CLASS_NAMES[c]].append(r)

RED_STATS = {}
for name in CLASS_NAMES:
    v = np.array(ratios[name])
    if not len(v):
        continue
    lv = collections.Counter(level_of(r) for r in v)
    RED_STATS[name] = {"n": int(len(v)), "median": round(float(np.median(v)), 3), "levels": dict(lv)}
    print(f"{name:8s}: {len(v)} kotak | median merah {np.median(v):.0%} | " + ", ".join(f"{k} {lv[k] / len(v):.0%}" for k in ("mentah", "setengah matang", "matang")))
plt.figure(figsize=(8, 4))
for name in CLASS_NAMES:
    plt.hist(ratios[name], bins=20, range=(0, 1), alpha=0.6, label=name)
for t in RED_LEVELS:
    plt.axvline(t, color="k", ls="--", lw=1)
plt.xlabel("fraksi piksel merah"); plt.ylabel("jumlah kotak"); plt.legend(); plt.title("Warna merah per kelas (label asli)")
plt.show()

In [ ]:
# 9) Ringkasan run: simpan lalu bandingkan antar percobaan (unduh run_summary.json dari tiap run)
import json

summary = {
    "task": "ripeness", "classes": CLASS_NAMES, "split_seed": SPLIT_SEED,
    "split_sizes": {s: len(SPLITS[s]) for s in SPLIT_NAMES},
    "negatives": sum(it["kind"] == "negatif" for it in ITEMS),
    "run": RUN, "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "hsv_h": HSV_H,
    "use_rfs": USE_RFS, "rfs_t": RFS_T, "train_entries_per_epoch": N_TRAIN,
    "test_map50": round(float(metrics.box.map50), 4), "test_map50_95": round(float(metrics.box.map), 4),
    "per_class": PER_CLASS, "confusion_test": CONFUSION, "false_alarm_rate_on_negatives": FP_RATE,
    "red_levels": RED_LEVELS, "red_ratio_by_class": RED_STATS,
}
Path("run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

In [ ]:
# 10) Kurva training dan confusion matrix
from IPython.display import Image as IPImage, display

for f in [BEST.parent.parent / "results.png", Path(metrics.save_dir) / "confusion_matrix_normalized.png"]:
    if f.exists():
        print(f.name)
        display(IPImage(filename=str(f)))

In [ ]:
# 11) Cek visual: prediksi pada 8 gambar test acak yang berisi buah
test_imgs = sorted(p for p in SPLITS["test"] if CLS_OF[p])
sample = random.Random(0).sample(test_imgs, min(8, len(test_imgs)))
results = best.predict([str(p) for p in sample], imgsz=IMGSZ, conf=0.4, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax in axes.ravel():
    ax.axis("off")
for ax, p, r in zip(axes.ravel(), sample, results):
    ax.imshow(r.plot()[:, :, ::-1])  # plot() menghasilkan BGR -> balik ke RGB
    ax.set_title(f"{p.name} | asli: " + "+".join(CLASS_NAMES[k] for k in sorted(CLS_OF[p])), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 12) Salin bobot ke /kaggle/working supaya mudah diunduh dari panel Output
shutil.copy(BEST, "best.pt")
print("Tersimpan:", Path("best.pt").resolve(), f"({Path('best.pt').stat().st_size / 1e6:.1f} MB)")

## Setelah training

1. Unduh `best.pt` dan `run_summary.json` (panel **Output** di sisi kanan, atau tab Output setelah Save & Run All).
2. Taruh keduanya di `kematangan/results/` pada folder proyek, lalu jalankan di laptop:

```
pip install -U ultralytics paho-mqtt     # paho-mqtt hanya jika memakai MQTT
python kematangan/realtime_kematangan.py                                              # webcam laptop
python kematangan/realtime_kematangan.py --source http://IP-ESP32:81/stream --fps 1 --no-show
```

Kalau hasil sel 8 menunjukkan ambang warna lain lebih cocok, ubah lewat `--levels`, mis. `--levels 0.25 0.65`.

## Catatan

- Dataset ini kecil dan sebagian besar foto dari internet (banyak foto stok berlatar polos). Ukuran sebenarnya tetap foto dari kamera yang akan dipakai di kebun. Tambahkan foto sendiri yang dilabeli `ripe`/`unripe` (mis. lewat Roboflow), lampirkan sebagai dataset tambahan, lalu latih ulang. Semua dataset yang kelasnya ada di `NAME_MAP` ikut dipakai otomatis.
- Untuk menambah kelas `setengah_matang` sungguhan, semua kotak di semua dataset harus dilabeli ulang dengan tiga kelas; mencampur dataset dua kelas dan tiga kelas akan membuat model bingung.

| Run | Perubahan | Test mAP50 | AP50 ripe | AP50 unripe | Alarm palsu tanpa buah |
|---|---|---|---|---|---|
| 1 | pengaturan awal | (isi) | (isi) | (isi) | (isi) |